In [1]:
import requests
import time
import pandas as pd

pd.set_option("display.max_rows", None) #option pour que Pandas arrête de raccourcir les affichages

# Récupération des données en utilisant l’API de Scryfall

In [2]:
headers = {
    'User-Agent': 'mtg-artifact-token-analyser/0.1',
    'Accept': 'application/json'
} #Scryfall exige un header pour les requêtes

base_url = 'https://api.scryfall.com/cards/search'
'''
Requête utilisant le système d’opérateur et le langage de REGEX de scryfall pour obtenir les cartes générant des jetons d’artefact sur la base de leur texte de règle.
On filtre sur les couleurs (URG), le format de jeu (commander), et on écarte les types exotiques (Dungeon, Stickers).

Problème de cette requête, il faut faire trois requêtes en une, à cause de la typologie du wording des textes de règles. On distingue 3 cas :
- le cas générique décrivant la création du jeton avec ses types, nom et effets ;
- le cas décrivant la création d’un jeton prédéfini dont la cartes ne donne que le nom (nécessité de vérifier la liste des jetons pertinents dans l’article 111.10 des Comprehensive Rules)
- les actions mots-clés créant un jeton d’artefact, à vérifier dans la section 701 des Comprehensive Rules.
'''
query = (
    "(o:/create.*artifact.*tokens?( named)?/ or "
    "o:/investigate|incubate|fabricate/ or "
    "o:/create.*(Gold|Lander|Map|Treasure|Heartwood|Food|Blood|Powerstone|Junk|Mutagen|Vibranium) tokens?.*/) "
    "id<=URG "
    "game:paper "
    "-t:Stickers "
    "-t:Dungeon "
    "f:commander"
) 

params = {
    "q": query,
}
r = requests.get(base_url, 
                 headers=headers, 
                 params=params,
                 timeout=30 #timeout pour éviter que la requête ne boucle sans réponse
                )

print(r.status_code)
print(r.url)


200
https://api.scryfall.com/cards/search?q=%28o%3A%2Fcreate.%2Aartifact.%2Atokens%3F%28+named%29%3F%2F+or+o%3A%2Finvestigate%7Cincubate%7Cfabricate%2F+or+o%3A%2Fcreate.%2A%28Gold%7CLander%7CMap%7CTreasure%7CHeartwood%7CFood%7CBlood%7CPowerstone%7CJunk%7CMutagen%7CVibranium%29+tokens%3F.%2A%2F%29+id%3C%3DURG+game%3Apaper+-t%3AStickers+-t%3ADungeon+f%3Acommander


# Création du DataFrame à partir du JSON de la requête

In [3]:
data = r.json()
print("Cartes trouvées :", data["total_cards"])
print("Cartes sur cette page :", len(data["data"]))
print("Autres pages :", data["has_more"])

Cartes trouvées : 693
Cartes sur cette page : 175
Autres pages : True


In [4]:
cards = [] # initialisation d’une liste vide pour récupérer les données des cartes obtenues via Requests
while base_url : #on boucle tant que base_url ne vaut pas None
    r = requests.get(
        base_url,
        headers = headers,
        params = params,
        timeout = 30
    )
    r.raise_for_status() #remontée des erreurs de réception 4xx et 5xx, fait planter le programme avant qu’on commence à analyser le JSON.

    data = r.json()
    cards.extend(data["data"]) #on concatène la table data du JSON à la liste cards pour gérer la pagination de la requête.

    base_url = data.get("next_page")
    params = None #Plus besoin de paramètres après la première itération de la boucle : les URLS des pages de résultats suivantes les fournissent déjà.
    time.sleep(0.15) #On ralentit les boucles pour ne pas se faire éjecter par Scryfall.

print(f"{len(cards)} cartes récupérées")
    

693 cartes récupérées


In [5]:
df = pd.DataFrame(cards) #Création d’un DataFrame pandas à partir de la liste des cartes.
df.shape
df.columns.tolist()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 693 entries, 0 to 692
Data columns (total 79 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   object             693 non-null    str    
 1   id                 693 non-null    str    
 2   oracle_id          693 non-null    str    
 3   multiverse_ids     693 non-null    object 
 4   mtgo_id            622 non-null    float64
 5   tcgplayer_id       693 non-null    int64  
 6   cardmarket_id      693 non-null    int64  
 7   name               693 non-null    str    
 8   lang               693 non-null    str    
 9   released_at        693 non-null    str    
 10  uri                693 non-null    str    
 11  scryfall_uri       693 non-null    str    
 12  layout             693 non-null    str    
 13  highres_image      693 non-null    bool   
 14  image_status       693 non-null    str    
 15  image_updated_at   693 non-null    str    
 16  image_uris         684 non-null    ob

In [6]:
'''
Réduction du nombre de champs pour ne garder que les plus pertinent, 
en faisant une copie pour ne pas toucher au jeu de données de base.
'''
columns = [
    "id",
    "oracle_id",
    "name",
    "mana_cost",
    "cmc",
    "type_line",
    "oracle_text",
    "colors",
    "color_identity",
    "all_parts",
    "legalities",
    "games",
    "reserved",
    "game_changer",
]

df_cards = df[columns].copy()

In [7]:
df_cards.shape
df_cards.columns.tolist()
df_cards.info()

<class 'pandas.DataFrame'>
RangeIndex: 693 entries, 0 to 692
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              693 non-null    str    
 1   oracle_id       693 non-null    str    
 2   name            693 non-null    str    
 3   mana_cost       684 non-null    str    
 4   cmc             693 non-null    float64
 5   type_line       693 non-null    str    
 6   oracle_text     671 non-null    str    
 7   colors          684 non-null    object 
 8   color_identity  693 non-null    object 
 9   all_parts       683 non-null    object 
 10  legalities      693 non-null    object 
 11  games           693 non-null    object 
 12  reserved        693 non-null    bool   
 13  game_changer    693 non-null    bool   
dtypes: bool(2), float64(1), object(5), str(6)
memory usage: 66.5+ KB


In [8]:
'''
Le champs all_parts est une tables contenant des objets JSON correspondant aux objets liées à la cartes.
Ce sont entre autres les jetons produit par la carte, c’est la seule donnée qu’on souhaite conserver ici.
'''
df_tokens = df_cards[["oracle_id", "name", "all_parts"]
    ].rename(columns={"name":"card_name"}).explode("all_parts") 
#on crée un DataFrame listant les liens entre jetons et cartes (une ligne par élément du all_parts, pas seulement les jetons), 
#en renommant le champ name de la carte au préalable (les jetons en ont un aussi)

df_tokens = df_tokens.dropna(
    subset = ["all_parts"]
).reset_index(drop=True)
#on élimine les lignes avec all_parts vide qu’on a pu créer avec explode.

parts = pd.json_normalize(
    df_tokens.pop("all_parts").tolist()
)
#on extrait les objets JSON de all_parts pour les aplatir en un DataFrame qui aura les même index que df_tokens (grâce à .reset_index(drop=True)

df_tokens = pd.concat(
    [df_tokens, parts],
    axis=1
)
#puis on concatène les deux DataFrames en collant les colonnes du deuxième au premier.

df_tokens = df_tokens[
    (df_tokens["component"] == "token") &
    (df_tokens["type_line"].str.contains(
        "Artifact", case=False, na=False
    ))
]
#enfin on réduit le nombre de ligne en ne gardant que les lignes correspondant à des jetons d’artefact

# Répartition des cartes du jeu de données en fonction des jetons qu’elles produisent

In [9]:
df_tokens.groupby("name")["oracle_id"].nunique().sort_values(ascending=False)

name
Treasure                  243
Clue                       79
Food                       77
Thopter                    44
Powerstone                 23
Blood                      18
Mutagen                    18
Construct                  17
Lander                     16
Servo                      15
Robot                      14
Golem                      12
Incubator // Phyrexian     11
Junk                       11
Phyrexian Golem             8
Map                         8
Myr                         7
Heartwood                   6
Soldier                     4
Clown Robot                 3
Illusion                    3
Phyrexian Myr               3
Vibranium                   3
Vehicle                     2
Gnome                       2
Necron Warrior              2
Drone                       2
Gold                        2
Axe                         1
Assembly-Worker             1
Borg                        1
Gremlin                     1
Feather                     1
Gobli

## Cas limite : des jetons qui n’apparaissent pas 
J’ai constaté qu’au moins un jeton que je connais est absent de la sortie alors qu’il devrait y être. Il s’agit du jeton Banana produit par la carte Kibo, Uktabi Prince. 

In [10]:
df_cards[df_cards["name"].str.contains("Kibo")]

,id,oracle_id,name,mana_cost,cmc,type_line,oracle_text,colors,color_identity,all_parts,legalities,games,reserved,game_changer
313,dcf7a304-4a45-42ed-9b1e-d36ae1963b66,0b9be4fa-5238-4afd-a9f6-f9022e67e5ab,"Kibo, Uktabi Prince",{2}{G},3.0,Legendary Creature — Monkey Noble,{T}: Each player creates a colorless artifact ...,[G],"[G, R]",NaN,"{'standard': 'not_legal', 'future': 'not_legal...",[paper],False,False


On constate que le champs all_part pour la carte de Kibo est vide. Il faut vérifier si c’est un cas isolé ou si d’autres cartes présentent cette anomalie.

En filtrant les cartes sur cette anomalie, on trouve 10 cas de figure incluant Kibo.
À part Bilbo, Fellow Conspirator qui a vraisemblablement été mal intégré dans la base de donnéés de Scryfall,
on constate que toutes les cartes posant problème sont des cartes pour lesquelles le jeton correspondant n’a jamais
été imprimé par l’éditeur. Et il s’agit à chaque fois d’un jeton unique.


In [11]:
df_cards[df_cards["all_parts"].isna()]

,id,oracle_id,name,mana_cost,cmc,type_line,oracle_text,colors,color_identity,all_parts,legalities,games,reserved,game_changer
33,5ff538f9-10b4-4327-aea6-a86759daf488,9155f78e-0ccc-4e08-ae79-ef5b2db1183f,Basalt Golem,{5},5.0,Artifact Creature — Golem,This creature can't be blocked by artifact cre...,[],[],NaN,"{'standard': 'not_legal', 'future': 'not_legal...","[paper, mtgo]",False,False
40,84a95ac5-e2f6-4cf5-bc5e-ac4410168c06,d273f87d-1271-49d2-a628-2d1bd888f56c,"Bilbo, Fellow Conspirator",{2}{G},3.0,Legendary Creature — Halfling Citizen,"If you would create a Food token, instead crea...",[G],[G],NaN,"{'standard': 'not_legal', 'future': 'not_legal...","[paper, mtgo, arena]",False,False
57,fec36f9a-cd66-4c01-ae3f-42fd82f0546b,f32d19d6-8ac1-4744-b2c2-5c9d4cd0da70,Bottle of Suleiman,{4},4.0,Artifact,"{1}, Sacrifice this artifact: Flip a coin. If ...",[],[],NaN,"{'standard': 'not_legal', 'future': 'not_legal...",[paper],False,False
141,548ff852-274d-4068-818d-58a883e74a5f,bffa9256-aff4-476f-86f4-4f4c9e57fa69,Diamond Kaleidoscope,{4},4.0,Artifact,"{3}, {T}: Create a 0/1 colorless Prism artifac...",[],[],NaN,"{'standard': 'not_legal', 'future': 'not_legal...","[paper, mtgo]",True,False
313,dcf7a304-4a45-42ed-9b1e-d36ae1963b66,0b9be4fa-5238-4afd-a9f6-f9022e67e5ab,"Kibo, Uktabi Prince",{2}{G},3.0,Legendary Creature — Monkey Noble,{T}: Each player creates a colorless artifact ...,[G],"[G, R]",NaN,"{'standard': 'not_legal', 'future': 'not_legal...",[paper],False,False
402,ba455e40-274a-4de7-bf9d-9b55b1896eda,288afdb9-708d-4a52-991a-e1b07f62ee99,Nuisance Engine,{3},3.0,Artifact,"{2}, {T}: Create a 0/1 colorless Pest artifact...",[],[],NaN,"{'standard': 'not_legal', 'future': 'not_legal...","[paper, mtgo]",False,False
537,36a3345d-1190-45f4-8191-897b4dcec376,ea7b0704-7715-4f8a-b32f-d2d2f3b36054,Spawning Pit,{2},2.0,Artifact,Sacrifice a creature: Put a charge counter on ...,[],[],NaN,"{'standard': 'not_legal', 'future': 'not_legal...","[paper, mtgo]",False,False
583,49c1a2b2-50f0-4ed0-bd8f-06cd6aada04f,85255c26-4e74-4cf0-91a6-78ddba5abdc6,Tetravus,{6},6.0,Artifact Creature — Construct,Flying\nThis creature enters with three +1/+1 ...,[],[],NaN,"{'standard': 'not_legal', 'future': 'not_legal...",[paper],False,False
661,89d43d64-eb3b-4152-90f7-2fca851499c6,69db9cb8-9152-4d1b-9ac9-16aac42c0908,Virtual Assistant,{1}{U},2.0,Artifact Creature — Illusion Advisor,Defender\nWhenever you cast a spell using team...,[U],[U],NaN,"{'standard': 'not_legal', 'future': 'not_legal...","[paper, mtgo, arena]",False,False
681,7bdb7e06-748c-4fc0-adc5-f8f2ec62029a,846bedc3-a1e0-4f11-a095-85b60ad482da,Wirefly Hive,{3},3.0,Artifact,"{3}, {T}: Flip a coin. If you win the flip, cr...",[],[],NaN,"{'standard': 'not_legal', 'future': 'not_legal...","[paper, mtgo]",False,False
